In [37]:
import tarfile
import gzip
import os
import json
import pickle
import urllib.parse
import re

In [38]:
import os
import gzip
import json
os.chdir("D:\Entity Aspect Linking\data\entity-aspect-linking-2020\collection")
path = "train-small.jsonl.gz"
data = {}
i = 1
with gzip.open(path, 'rt', encoding='UTF-8') as zipfile:
    for line in zipfile:
        my_object = json.loads(line)
        data[str(i)] = my_object
        i = i + 1


In [39]:
def decode(s):
    s = s.replace("enwiki:","")
    return urllib.parse.unquote(s)

In [40]:
def clean_aspect(s):
    s = decode(s)
    pattern = re.compile(r'.*?/')
    return re.sub(pattern, '', s)

In [41]:
def make_entdict(data, n = 10, israndom = False, ind = []):
    ent_data = []
    if israndom == True:
        it = list(map(int, ind))
    else:
        it = range(n)
    i = 0
    for i in it:
        temp = {}
        #if i == 5498:
            #print(True)
        ent = data[str(i+1)]
        #print(ent['context']['target_entity'])
        temp["id"] = str(i)
        temp["target_entity"] = decode(ent['context']['target_entity'])
        temp["paragraph"] = ent['context']['paragraph']['content']
        temp["entities"] = []                        
        k = 0
        for j in ent['context']['paragraph']['entities']:
            ent = {}                          
            if not j["target_mention"]:
                ent['eid'] = str(i) + str(k)
                ent['entity'] = j['entity_name']
                ent['mention'] = j['mention']
                temp["entities"].append(ent)
            k += 1
        ent_data.append(temp)

    return ent_data

In [42]:
ent = make_entdict(data, n = len(data.keys()))

In [43]:
def get_aspectdict(data, n = 10, israndom = False, ind = []):
    aspects = []
    k = 0
    if israndom == True:
        it = list(map(int, ind))
    else:
        it = range(n)
    for i in it:
        temp = {}
        #if i == 5498:
            #print(True)
        ent = data[str(i + 1)]
        temp['id'] = str(i)
        temp['true_aspect'] = clean_aspect(ent['true_aspect'])
        candasp = []
        j = 0
        for cand in ent['candidate_aspects']:
            casp = {}
            casp['id'] = 'A' + str(i) + str(j)
            casp['aspect_name'] = cand['aspect_name']
            casp['section_heading'] = cand['location']['section_headings']
            casp['content'] = cand['aspect_content']['content']
            ent = []
            for e in cand['aspect_content']['entities']:
                temp2 = {}
                temp2['entity_name'] = e['entity_name']
                temp2['eid'] = 'E' + str(k)
                temp2['mention'] = e['mention']
                ent.append(temp2)
                k+= 1
            casp['entities'] = ent
            candasp.append(casp)
            j += 1
        temp['candidate_aspects'] = candasp
        aspects.append(temp)
    return aspects
            

In [44]:
asp = get_aspectdict(data, n = len(data.keys()))

In [51]:
os.chdir("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\picklefiles")

In [52]:
final = []
for i in range(len(ent)):
    final.append((ent[i], asp[i]))

In [53]:
with open("extract_trainsmall.pkl", 'wb') as f:
    

5498